# 02 — Rebuild DualPath Dataset from Original BDD100K ZIPs

This notebook rebuilds the dataset used by the current DualPathNet project directly from the original files in `EcoCAR/downloads`:

- `bdd100k_labels.zip`
- `bdd100k_images_100k.zip`
- `bdd100k_seg_maps.zip`

It avoids the older preprocessing notebook entirely.

## 1 — Setup

In [ ]:
!pip install -q pyyaml tqdm
import os, sys, json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
PROJECT_DIR = os.path.join(ECOCAR_ROOT, 'DETR_GeoLane_pipeline')
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(f'Project not found: {PROJECT_DIR}')
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('PROJECT_DIR =', PROJECT_DIR)
print('ECOCAR_ROOT =', ECOCAR_ROOT)

## 2 — Choose paths

In [ ]:
from src.config import DOWNLOADS_DIR, RAW_DATASET_ROOT, PATHS_CONFIG

DOWNLOADS = DOWNLOADS_DIR
RAW_ROOT = RAW_DATASET_ROOT
OUTPUT_ROOT = os.path.join(ECOCAR_ROOT, 'datasets', 'bdd100k_yolo')
IMAGE_MODE = 'symlink'   # 'symlink' or 'copy'
FORCE_EXTRACT = False

print('DOWNLOADS  =', DOWNLOADS)
print('RAW_ROOT   =', RAW_ROOT)
print('OUTPUT_ROOT=', OUTPUT_ROOT)
print('PATHS_CONFIG =', PATHS_CONFIG)
for name in ['bdd100k_labels.zip', 'bdd100k_images_100k.zip', 'bdd100k_seg_maps.zip']:
    p = os.path.join(DOWNLOADS, name)
    print(name, 'FOUND' if os.path.isfile(p) else 'MISSING', p)

## 3 — Rebuild the dataset

In [ ]:
from src.data_prep import rebuild_dualpath_dataset

summary = rebuild_dualpath_dataset(
    downloads_dir=DOWNLOADS,
    raw_root=RAW_ROOT,
    output_root=OUTPUT_ROOT,
    image_mode=IMAGE_MODE,
    force_extract=FORCE_EXTRACT,
)

print(summary)

## 4 — Verify the rebuilt labels

In [ ]:
from src.data_prep import summarize_label_ids
from src.config import VEHICLE_CLASSES

train_label_dir = os.path.join(OUTPUT_ROOT, 'labels', 'train')
val_label_dir = os.path.join(OUTPUT_ROOT, 'labels', 'val')

print('Train raw label IDs:', summarize_label_ids(train_label_dir, max_files=3000))
print('Val raw label IDs:  ', summarize_label_ids(val_label_dir, max_files=1000))

print('
Train vehicle counts:')
for name in VEHICLE_CLASSES:
    print(f'  {name:<12}: {summary.counts_train.get(name, 0):>8,}')

print('
Val vehicle counts:')
for name in VEHICLE_CLASSES:
    print(f'  {name:<12}: {summary.counts_val.get(name, 0):>8,}')

## 5 — Quick spot check

In [ ]:
import random

def read_first_lines(txt_path, n=5):
    if not os.path.isfile(txt_path):
        return []
    with open(txt_path, 'r') as f:
        return [line.strip() for _, line in zip(range(n), f)]

train_images = sorted(os.listdir(os.path.join(OUTPUT_ROOT, 'images', 'train')))
random.seed(42)
for fname in random.sample(train_images, min(5, len(train_images))):
    stem = os.path.splitext(fname)[0]
    txt_path = os.path.join(OUTPUT_ROOT, 'labels', 'train', stem + '.txt')
    print('
' + '='*80)
    print('image:', fname)
    print('label exists:', os.path.isfile(txt_path))
    print('first lines:', read_first_lines(txt_path, n=5))

## 6 — Copy a local fast-I/O mirror (optional)

In [ ]:
LOCAL_DS = '/content/bdd100k_yolo'
if not os.path.exists(LOCAL_DS):
    os.symlink(OUTPUT_ROOT, LOCAL_DS, target_is_directory=True)
print('LOCAL_DS =', LOCAL_DS)
print('images/train exists:', os.path.isdir(os.path.join(LOCAL_DS, 'images', 'train')))
print('labels/train exists:', os.path.isdir(os.path.join(LOCAL_DS, 'labels', 'train')))

## 7 — What to do next

In [ ]:
print('1) Open notebook 00_dualpath_pipeline.ipynb')
print('2) Keep cfg.dataset_root pointing to /content/bdd100k_yolo or EcoCAR/datasets/bdd100k_yolo')
print('3) Train again — motorcycle and bicycle should now be present if the original JSON contains them')
print('4) Lane labels will be discovered from paths_config.yaml written during rebuild')